# Module 5: Train a Wordle Agent with GRPO

Fine-tune Qwen3-1.7B to play Wordle using GRPO (Group Relative Policy Optimization) via TRL and OpenEnv.

**Time:** ~90 min (training) · **Difficulty:** Advanced · **GPU:** A100 required (Colab Pro or similar)

Based on the [TRL OpenEnv Wordle example](https://github.com/huggingface/trl/blob/main/examples/notebooks/openenv_wordle_grpo.ipynb).

In [1]:
# Requires GPU (A100 recommended). Run on Colab Pro or similar.
!pip install -Uq "trl>=0.17.0" openenv-core transformers datasets accelerate vllm trackio
!git clone --depth=1 -q https://github.com/meta-pytorch/OpenEnv.git 2>/dev/null || true

import sys, os
repo = os.path.abspath('OpenEnv')
for p in [repo, os.path.join(repo, 'src')]:
    if p not in sys.path:
        sys.path.insert(0, p)
print("Setup complete!")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.9/87.9 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 531.0/531.0 kB 17.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 174.1/174.1 kB 20.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 37.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 433.2/433.2 MB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 192.6/192.6 kB 675.6 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 410.6 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.8/7.8 MB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.0/111.0 kB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.4/45.4 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.9/3.9 MB 71.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
from huggingface_hub import notebook_login
notebook_login()

## 1. Initialize the Environment

Connect to the TextArena Wordle environment hosted on HF Spaces.

> **For production use:** Duplicate the Space to your own account to avoid concurrency limits.

In [ ]:
from envs.textarena_env import TextArenaEnv

textarena_url = 'https://burtenshaw-textarena.hf.space'  # Duplicate this Space for production use!

# Verify connection
with TextArenaEnv(base_url=textarena_url).sync() as _check:
    result = _check.reset()
    print(f'Connected to: {textarena_url}')
    print(f'Prompt preview: {str(result.observation.prompt)[:100]}...')

# Create a persistent sync client for training.
# A single WebSocket connection is reused across all rollouts instead of
# opening/closing one per episode, which matters at training throughput.
env = TextArenaEnv(base_url=textarena_url)
sync_env = env.sync()
sync_env.connect()
print('Persistent training connection established.')

## 2. Init Model and Tokenizer

In [ ]:
from transformers import AutoTokenizer

model_name = "Qwen/Qwen3-1.7B"
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token
print(f"Model: {model_name}")

## 3. Define the System Prompt

In [ ]:
system_prompt = """
You are an expert Wordle solver with deep knowledge of English vocabulary, letter frequency patterns, and optimal guessing strategies.

## GAME RULES

1. The target is a 5-letter English word
2. You have 6 attempts to guess the correct word
3. After each guess, you receive color-coded feedback:
   - GREEN: Letter is correct and in the correct position
   - YELLOW: Letter is in the word but in the wrong position
   - GRAY: Letter is not in the word at all
4. All guesses must be valid 5-letter English words
5. You cannot reuse a word you've already guessed

## RESPONSE FORMAT

Only respond with your next guess in square brackets, e.g., [crane].

## STRATEGIC APPROACH

Do not repeat the same guess twice.

### Opening Strategy
- Start with words rich in common vowels (A, E, I, O, U) and consonants (R, S, T, L, N)
- Optimal starters: CRANE, SLATE, STARE, AROSE, IRATE

### Mid-Game Strategy
- Use confirmed GREEN letters in their correct positions
- Place YELLOW letters in different positions than where they appeared
- Eliminate GRAY letters from consideration

## YOUR GOAL

Solve the Wordle in as few guesses as possible.
"""

## 4. Helper Functions

In [ ]:
def make_user_prompt(prompt_text, messages):
    """Build a structured prompt from game state and message history."""
    history = format_history(messages)
    prompt_section = prompt_text.strip() if prompt_text.strip() else "Wordle-v0"
    history_section = history if history else "[PROMPT] Awaiting first feedback."
    return (
        f"Game prompt:\n{prompt_section}\n\n"
        f"Conversation so far:\n{history_section}\n\n"
        "Reply with your next guess enclosed in square brackets."
    )


def format_history(messages):
    """Format message history with category tags."""
    lines = []
    for message in messages:
        tag = message.category or "MESSAGE"
        content = message.content.strip()
        if content:
            lines.append(f"[{tag}] {content}")
    return "\n".join(lines)


def scale_repetition_score(previous_occurrences, max_occurrences):
    """Scale repetition penalty: 1.0 = novel guess, 0.0 = fully repeated."""
    if max_occurrences == 0:
        return 0.0
    return (max_occurrences - previous_occurrences) / max_occurrences


print("Helper functions defined.")

## 5. Define the Rollout Function

The rollout function plays one full Wordle game per prompt. It's called by `GRPOTrainer` during training.

In [ ]:
from collections import defaultdict
from envs.textarena_env.models import TextArenaAction
from envs.textarena_env.rewards import extract_feedback_counts, extract_guess, extract_wordle_feedback
from trl.experimental.openenv import generate_rollout_completions


def rollout_once(trainer, sync_env, tokenizer, dataset_prompt, system_prompt, max_turns):
    """Execute one full Wordle episode using an already-connected sync client."""
    result = sync_env.reset()
    observation = result.observation

    prompt_ids = []
    completion_ids = []
    logprobs = []
    green_scores = []
    yellow_scores = []
    repetition_scores = []
    correct_scores = []
    guess_counts = defaultdict(int)

    for _turn in range(max_turns):
        if result.done:
            break

        base_prompt = observation.prompt or dataset_prompt
        user_prompt = make_user_prompt(base_prompt, observation.messages)
        messages = [
            {'role': 'system', 'content': system_prompt},
            {'role': 'user', 'content': user_prompt},
        ]
        prompt_text = tokenizer.apply_chat_template(
            messages,
            add_generation_prompt=True,
            tokenize=False,
            enable_thinking=False,
        )

        rollout_outputs = generate_rollout_completions(trainer, [prompt_text])[0]
        prompt_ids.extend(rollout_outputs['prompt_ids'])
        completion_ids.extend(rollout_outputs['completion_ids'])
        logprobs.extend(rollout_outputs['logprobs'])
        completion_text = rollout_outputs.get('text') or tokenizer.decode(
            rollout_outputs['completion_ids'], skip_special_tokens=True
        )

        guess = extract_guess(completion_text)
        result = sync_env.step(TextArenaAction(message=guess))
        observation = result.observation
        correct_score = float(result.reward or 0.0)
        feedback = extract_wordle_feedback(observation)

        previous_occurrences = guess_counts[guess]
        repetition_score = max(0.0, 1.0 - previous_occurrences)
        guess_counts[guess] += 1

        if not feedback:
            green_score, yellow_score = 0.0, 0.0
        else:
            green_count, yellow_count = extract_feedback_counts(feedback)
            green_score = green_count / 5.0
            yellow_score = yellow_count / 5.0

        repetition_scores.append(repetition_score)
        green_scores.append(green_score)
        yellow_scores.append(yellow_score)
        correct_scores.append(correct_score)

    return {
        'prompt_ids': prompt_ids,
        'completion_ids': completion_ids,
        'logprobs': logprobs,
        'correct_reward': correct_scores[-1] if correct_scores else 0.0,
        'green_reward': green_scores[-1] if green_scores else 0.0,
        'yellow_reward': yellow_scores[-1] if yellow_scores else 0.0,
        'repetition_reward': repetition_scores[-1] if repetition_scores else 0.0,
    }


def rollout_func(prompts, trainer=None):
    """Rollout function called by GRPOTrainer. Uses the module-level sync_env."""
    episode_prompt_ids = []
    episode_completion_ids = []
    episode_logprobs = []
    correctness_rewards = []
    green_rewards = []
    yellow_rewards = []
    repetition_rewards = []

    for prompt_text in prompts:
        episode = rollout_once(
            trainer=trainer,
            sync_env=sync_env,     # Persistent connection — no reconnect per episode
            tokenizer=tokenizer,
            dataset_prompt=prompt_text,
            system_prompt=system_prompt,
            max_turns=6,
        )
        episode_prompt_ids.append(episode['prompt_ids'])
        episode_completion_ids.append(episode['completion_ids'])
        episode_logprobs.append(episode['logprobs'])
        correctness_rewards.append(episode['correct_reward'])
        green_rewards.append(episode['green_reward'])
        yellow_rewards.append(episode['yellow_reward'])
        repetition_rewards.append(episode['repetition_reward'])

    return {
        'prompt_ids': episode_prompt_ids,
        'completion_ids': episode_completion_ids,
        'logprobs': episode_logprobs,
        'correct_reward': correctness_rewards,
        'green_reward': green_rewards,
        'yellow_reward': yellow_rewards,
        'repetition_reward': repetition_rewards,
    }


print('Rollout functions defined.')

## 6. Define Reward Functions

Four reward signals for richer gradient information.

In [ ]:
def reward_correct(completions, **kwargs):
    rewards = kwargs.get("correct_reward")
    return [float(r) for r in rewards] if rewards else [0.0] * len(completions)

def reward_greens(completions, **kwargs):
    rewards = kwargs.get("green_reward")
    return [float(r) for r in rewards] if rewards else [0.0] * len(completions)

def reward_yellows(completions, **kwargs):
    rewards = kwargs.get("yellow_reward")
    return [float(r) for r in rewards] if rewards else [0.0] * len(completions)

def reward_repetition(completions, **kwargs):
    rewards = kwargs.get("repetition_reward")
    return [float(r) for r in rewards] if rewards else [0.0] * len(completions)

print("Reward functions: correct, greens, yellows, repetition")

## 7. Create Dataset

In [ ]:
from datasets import Dataset

dataset_size = 1000
dataset = Dataset.from_dict({"prompt": ["Play Wordle like an expert."] * dataset_size})
print(f"Dataset: {len(dataset)} prompts")

## 8. Configure GRPO Training

In [ ]:
from trl import GRPOConfig

output_dir = "wordle-grpo-Qwen3-1.7B"

grpo_config = GRPOConfig(
    num_train_epochs=1,
    learning_rate=5e-6,
    gradient_accumulation_steps=64,
    per_device_train_batch_size=1,
    warmup_steps=20,
    num_generations=2,
    max_completion_length=8,
    max_prompt_length=1400,
    use_vllm=True,
    vllm_mode="colocate",
    vllm_gpu_memory_utilization=0.1,
    output_dir=output_dir,
    report_to="trackio",
    trackio_space_id=output_dir,
    logging_steps=1,
    save_steps=10,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    push_to_hub=True,
)

print(f"Output: {output_dir}")
print(f"vLLM mode: colocate (generation + training on same GPU)")

## 9. Create Trainer and Train

In [ ]:
from trl import GRPOTrainer

trainer = GRPOTrainer(
    model=model_name,
    processing_class=tokenizer,
    reward_funcs=[
        reward_correct,
        reward_greens,
        reward_yellows,
        reward_repetition,
    ],
    train_dataset=dataset,
    args=grpo_config,
    rollout_func=rollout_func,
)

In [ ]:
# Check GPU before training
import torch
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU: {gpu_stats.name} — {max_memory} GB total, {start_gpu_memory} GB reserved")

In [ ]:
# Train (~90 minutes on A100)
trainer_stats = trainer.train()

In [ ]:
# Memory stats after training
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_for_training = round(used_memory - start_gpu_memory, 3)

print(f"Training time: {round(trainer_stats.metrics['train_runtime']/60, 1)} minutes")
print(f"Peak memory: {used_memory} GB ({round(used_memory/max_memory*100, 1)}% of {max_memory} GB)")
print(f"Memory for training: {used_for_training} GB")

## 10. Save and Push

In [ ]:
# Close the persistent environment connection before saving
sync_env.close()

trainer.save_model(output_dir)
trainer.push_to_hub()
print(f'Model saved to {output_dir} and pushed to Hub.')

## 11. Evaluate: Play Wordle with the Trained Model

In [ ]:
from transformers import AutoModelForCausalLM
from envs.textarena_env.models import TextArenaAction
from envs.textarena_env.rewards import extract_guess

# Load the fine-tuned model (replace with your HF repo id if you pushed)
fine_tuned_model = AutoModelForCausalLM.from_pretrained(
    output_dir, torch_dtype='auto', device_map='auto'
)


def play_wordle(sync_env, model, tokenizer, max_turns=6):
    """Play one Wordle game and print each turn."""
    result = sync_env.reset()
    observation = result.observation
    print(f'Prompt: {observation.prompt[:100]}...')

    for turn in range(max_turns):
        if result.done:
            break

        user_prompt = make_user_prompt(observation.prompt, observation.messages)
        messages = [
            {'role': 'system', 'content': system_prompt},
            {'role': 'user', 'content': user_prompt},
        ]
        prompt_text = tokenizer.apply_chat_template(
            messages, add_generation_prompt=True,
            tokenize=False, enable_thinking=False,
        )

        model_inputs = tokenizer([prompt_text], return_tensors='pt').to(model.device)
        generated_ids = model.generate(**model_inputs, max_new_tokens=512)
        output_ids = generated_ids[0][len(model_inputs.input_ids[0]):]
        generated_text = tokenizer.decode(output_ids, skip_special_tokens=True)
        guess = extract_guess(generated_text)

        print(f'\nTurn {turn + 1}: {guess}')
        result = sync_env.step(TextArenaAction(message=guess))
        observation = result.observation
        for msg in observation.messages:
            print(f'  [{msg.category}] {msg.content}')

    print(f'\nResult: reward={result.reward}, done={result.done}')


# Evaluation uses a fresh per-game context (not the training connection)
eval_env = TextArenaEnv(base_url=textarena_url)
with eval_env.sync() as eval_sync:
    play_wordle(eval_sync, fine_tuned_model, tokenizer)

## Summary

What you did:
1. Connected to the TextArena Wordle environment via OpenEnv
2. Defined a system prompt, rollout function, and 4 reward signals
3. Trained Qwen3-1.7B with GRPO for ~90 minutes on an A100
4. Evaluated the trained model on live Wordle games

The key insight: **OpenEnv makes the environment a plug-in.** Swap Wordle for any other OpenEnv environment — your Module 4 word game, a coding environment, a math problem — and the training pipeline stays the same.

### What's next

- **Improve the model:** Longer training, larger models, better reward shaping
- **Build your own environment:** Use Module 4's pattern, plug it into this pipeline
- **Scale up:** See the [Scaling appendix](../README.md#bonus-scaling-openenv) for multi-container deployment
- **Explore the Hub:** Browse [openenv environments](https://huggingface.co/collections/openenv/environment-hub) for inspiration

# Task
Fine-tune a "Qwen/Qwen3-1.7B" model using GRPO to play Wordle in the "https://burtenshaw-textarena.hf.space" TextArena environment, saving the trained model to the "wordle-grpo-Qwen3-1.7B" directory, and then evaluate its performance.

## Environment Setup

### Subtask:
Execute the cell to install necessary Python packages (`trl`, `openenv-core`, `transformers`, `datasets`, `accelerate`, `vllm`, `trackio`) and clone the OpenEnv repository. This prepares the Colab environment for training.


## Hugging Face Login

### Subtask:
Run the `notebook_login()` function to authenticate with Hugging Face Hub, which is required for accessing models and pushing the fine-tuned model.


## Initialize Wordle Environment

### Subtask:
Connect to the TextArena Wordle environment hosted on Hugging Face Spaces and establish a persistent sync client for training rollouts.


**Reasoning**:
The subtask requires connecting to the TextArena Wordle environment and establishing a persistent sync client. This code block performs all the necessary steps as outlined in the instructions.



In [5]:
from envs.textarena_env import TextArenaEnv

textarena_url = 'https://burtenshaw-textarena.hf.space'  # Duplicate this Space for production use!

# Verify connection
with TextArenaEnv(base_url=textarena_url).sync() as _check:
    result = _check.reset()
    print(f'Connected to: {textarena_url}')
    print(f'Prompt preview: {str(result.observation.prompt)[:100]}...')

# Create a persistent sync client for training.
# A single WebSocket connection is reused across all rollouts instead of
# opening/closing one per episode, which matters at training throughput.
env = TextArenaEnv(base_url=textarena_url)
sync_env = env.sync()
sync_env.connect()
print('Persistent training connection established!')

Connected to: https://burtenshaw-textarena.hf.space
Prompt preview: ...
Persistent training connection established!


**Reasoning**:
The previous code failed because of a `ModuleNotFoundError` within the `OpenEnv/envs/textarena_env/rewards.py` file, specifically related to importing `TextArenaAction` and `TextArenaObservation`. This is likely due to an incorrect relative import path within the cloned repository. The fix is to modify the `rewards.py` file to use a correct relative import.



In [4]:
import re

rewards_file_path = "/content/OpenEnv/envs/textarena_env/rewards.py"

# Read the content of the rewards.py file
with open(rewards_file_path, "r") as f:
    content = f.read()

# Replace the problematic import block with a correct relative import
# This regex specifically targets the try-except block identified in the traceback
fixed_content = re.sub(
    r"try:\s*from textarena_env.models import TextArenaAction, TextArenaObservation\s*except ImportError:\s*from models import TextArenaAction, TextArenaObservation",
    "from .models import TextArenaAction, TextArenaObservation",
    content,
    flags=re.DOTALL
)

# Write the modified content back to the rewards.py file
with open(rewards_file_path, "w") as f:
    f.write(fixed_content)

print(f"Fixed imports in {rewards_file_path}")

Fixed imports in /content/OpenEnv/envs/textarena_env/rewards.py


## Load Model and Tokenizer

### Subtask:
Initialize the `Qwen/Qwen3-1.7B` model's tokenizer. This tokenizer will be used for processing prompts and completions during training.


**Reasoning**:
The subtask requires initializing the tokenizer for the 'Qwen/Qwen3-1.7B' model, setting its padding token, and printing the model name. This can be accomplished by importing `AutoTokenizer`, defining the model name, loading the tokenizer from pre-trained, setting `pad_token` to `eos_token`, and then printing the model name.



In [6]:
from transformers import AutoTokenizer

model_name = "Qwen/Qwen3-1.7B"
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token
print(f"Model: {model_name}")

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

Model: Qwen/Qwen3-1.7B


**Reasoning**:
The previous code block successfully initialized the tokenizer and printed the model name. The next logical step in the notebook's flow is to define the system prompt for the Wordle agent, as outlined in section 3 of the original notebook.



In [7]:
## 3. Define the System Prompt

## 3. Define the System Prompt

**Reasoning**:
Following the `markdown_block` for '3. Define the System Prompt', the next step is to define the actual `system_prompt` variable with its multi-line string content, as seen in the original notebook.



In [8]:
system_prompt = """
You are an expert Wordle solver with deep knowledge of English vocabulary, letter frequency patterns, and optimal guessing strategies.

## GAME RULES

1. The target is a 5-letter English word
2. You have 6 attempts to guess the correct word
3. After each guess, you receive color-coded feedback:
   - GREEN: Letter is correct and in the correct position
   - YELLOW: Letter is in the word but in the wrong position
   - GRAY: Letter is not in the word at all
4. All guesses must be valid 5-letter English words
5. You cannot reuse a word you've already guessed

## RESPONSE FORMAT

Only respond with your next guess in square brackets, e.g., [crane].

## STRATEGIC APPROACH

Do not repeat the same guess twice.

### Opening Strategy
- Start with words rich in common vowels (A, E, I, O, U) and consonants (R, S, T, L, N)
- Optimal starters: CRANE, SLATE, STARE, AROSE, IRATE

### Mid-Game Strategy
- Use confirmed GREEN letters in their correct positions
- Place YELLOW letters in different positions than where they appeared
- Eliminate GRAY letters from consideration

## YOUR GOAL

Solve the Wordle in as few guesses as possible.
"""